# 02 — Attack-Family Classification

## Research Question
Can telemetry identify which family of attack occurred, rather than only detecting that an attack occurred?

## Hypothesis
Different adversarial mechanisms induce distinguishable telemetry signatures across execution traces.

## Input Data
- `episode_df_all`

## Prediction/Analysis Target
- Multiclass labels: `attack_type` and optionally `Tier`

## Validation Protocol
Leave-one-seed-out and cross-validation; macro-F1 and balanced accuracy are prioritized.

## Expected Paper Artifact
- Attack-family classification table
- Confusion matrix
- TDS summary


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    make_scorer,
)

from evaluation import (
    load_jsonl, 
    evaluate_binary_prediction, 
    evaluate_multiclass_attack_type,
    fit_rf_feature_importance,
    evaluate_multiclass_feature_group_ablation,
    add_akc_phase_from_attack_type,
    normalize_seed_column,
    infer_telemetry_features,
    prepare_akc_phase_df,
    run_akc_phase_leave_one_seed,
    summarize_multiclass_results,
    collect_akc_phase_predictions_leave_one_seed,
    plot_confusion_matrix_from_predictions,
    infer_early_akc_features,
    classification_report,
    run_temporal_akc_phase_classification,
    plot_temporal_akc_curves,
    adversarial_fragmentation_index,
    risk_conditioned_fragmentation,
    semantic_fragmentation_proxy,
    add_benign_normalized_fragmentation,
    evaluate_leave_one_seed_out,
    TELEMETRY_NUMERIC,
    evaluate_feature_group_ablation,
    FEATURE_GROUPS,
    evaluate_leave_one_attack_out,
    compute_permutation_importance,
    run_operational_only_leave_one_seed,
    summarize_results,
    plot_exp1_operational_comparison,
    load_tamas_jsonl,
    build_early_df_from_agent_outputs,
    run_early_detection_leave_one_seed,
    plot_early_detection_curve,
    EARLY_FEATURES_CANDIDATES,
    validate_early_df_for_attack_analysis,
    run_early_detection_by_attack_type,
    summarize_early_detection_by_attack,
    plot_metric_curves_by_attack,
    compute_temporal_gain,
    infer_numeric_features,
    keep_existing_numeric,
    run_early_detection_feature_ablation,
    plot_ablation_curves,
    telemetry_guardrail_decision,
    run_akc_phase_leave_one_attack_type_out,
)
from datasets.tamas.tamas_features import build_all_feature_tables

RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

scenario = "education"
architecture = "centralized_tamas"
CONDITIONS = [
    "benign",
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]
SEEDS = [1, 2, 3]
MODEL_NAMES = [
    "ticlazau/meta-llama-3.1-8b-instruct:latest",
    # "qwen3",
]

In [ ]:

# Load processed telemetry tables generated by 00_setup_and_feature_extraction.ipynb.
# If files are not available yet, run notebook 00 first.
PROCESSED_DIR = Path("results/tamas/processed")

for _name in ["episode_df_all", "agent_df_all", "early_df_all", "df_raw_all"]:
    _path = PROCESSED_DIR / f"{_name}.parquet"
    if _path.exists():
        globals()[_name] = pd.read_parquet(_path)
        print(f"Loaded {_name}: {globals()[_name].shape}")
    else:
        print(f"Missing {_path}; run 00_setup_and_feature_extraction.ipynb if this notebook needs it.")


### Classification - attack_type

In [ ]:
multiclass_results = []
for feature_set in ["baseline", "telemetry"]:
    for model_kind in ["logreg", "rf"]:
        res = evaluate_multiclass_attack_type(
            episode_df_all,
            feature_set=feature_set,
            model_kind=model_kind,
        )
        if res:
            multiclass_results.append(res)

multiclass_results_df = pd.DataFrame(multiclass_results)
display(multiclass_results_df)

### Feature Importance - attack_type

In [ ]:
imp = fit_rf_feature_importance(episode_df_all, "attack_type")

if not imp.empty:
    # display(imp.head(15))
    top = imp.head(12).iloc[::-1]
    plt.figure(figsize=(8, 5))
    plt.barh(top["feature"], top["importance"])
    plt.title(f"Feature importance: attack_type")
    plt.tight_layout()
    plt.show()


### Feature Ablation - attack_type

In [ ]:
ablation_attack_type_df = evaluate_multiclass_feature_group_ablation(
    episode_df_all,
    target_col="attack_type",
    model_kind="rf",
)

display(
    ablation_attack_type_df.sort_values(
        "balanced_accuracy_mean",
        ascending=False,
    )
)

In [ ]:
def evaluate_multiclass_attack_type(
    df,
    model_kind="rf",
    min_samples_per_class=2,
):
    df = df.copy()
    df = df[df["Tier"].notna()].copy()

    counts = df["Tier"].value_counts()
    valid_classes = counts[counts >= min_samples_per_class].index
    df = df[df["Tier"].isin(valid_classes)].copy()

    if df.empty or df["Tier"].nunique() < 2:
        print("Insufficient classes for multiclass evaluation.")
        return None

    y = df["Tier"].astype(str)
    cols = df.columns.tolist()
    cols.remove("Tier")

    transformers = [
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), cols)
    ]

    pre = ColumnTransformer(transformers)

    if model_kind == "logreg":
        clf = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            # multi_class="auto",
        )
    elif model_kind == "rf":
        clf = RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            class_weight="balanced_subsample",
            min_samples_leaf=2,
        )
    else:
        raise ValueError(f"Unsupported model_kind: {model_kind}")

    pipe = Pipeline([
        ("pre", pre),
        ("clf", clf),
    ])

    min_class = y.value_counts().min()
    cv = StratifiedKFold(
        n_splits=min(5, min_class),
        shuffle=True,
        random_state=42,
    )

    scoring = {
        "balanced_accuracy": "balanced_accuracy",
        "f1_macro": make_scorer(f1_score, average="macro"),
        "f1_weighted": make_scorer(f1_score, average="weighted"),
    }

    scores = cross_validate(
        pipe,
        df,
        y,
        cv=cv,
        scoring=scoring,
        error_score=np.nan,
    )

    return {
        "target": "Tier",
        "model": model_kind,
        "n": len(df),
        "num_classes": y.nunique(),
        "balanced_accuracy_mean": float(np.nanmean(scores["test_balanced_accuracy"])),
        "f1_macro_mean": float(np.nanmean(scores["test_f1_macro"])),
        "f1_weighted_mean": float(np.nanmean(scores["test_f1_weighted"])),
    }

evaluate_multiclass_attack_type(
    df_subset_label,
    model_kind="logreg",
    min_samples_per_class=2,
)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, balanced_accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

def compute_tds_by_model(
    df,
    features,
    target_col="is_attack",
    n_splits=5,
):

    g = df.copy()

    X = g[features]
    y = g[target_col]

    min_class = y.value_counts().min()
    cv_splits = min(n_splits, min_class)

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            class_weight="balanced_subsample",
            min_samples_leaf=2,
        )),
    ])

    cv = StratifiedKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=42,
    )

    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring = {
            "balanced_accuracy": make_scorer(balanced_accuracy_score),
            "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
            "f1_weighted": make_scorer(f1_score, average="weighted", zero_division=0),
            "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
            "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
        },
        error_score="raise",
    )
    rows = {
        "n": len(g),
        "target_col": target_col,
        "num_classes": y.nunique(),
        "majority_class": y.value_counts().idxmax(),
        "majority_class_rate": y.value_counts(normalize=True).max(),

        "tds_bal_acc_mean": scores["test_balanced_accuracy"].mean(),
        "tds_bal_acc_std": scores["test_balanced_accuracy"].std(),

        "tds_f1_macro_mean": scores["test_f1_macro"].mean(),
        "tds_f1_macro_std": scores["test_f1_macro"].std(),

        "tds_f1_weighted_mean": scores["test_f1_weighted"].mean(),
        "tds_f1_weighted_std": scores["test_f1_weighted"].std(),

        "tds_precision_macro_mean": scores["test_precision_macro"].mean(),
        "tds_precision_macro_std": scores["test_precision_macro"].std(),

        "tds_recall_macro_mean": scores["test_recall_macro"].mean(),
        "tds_recall_macro_std": scores["test_recall_macro"].std(),
    }

    return pd.DataFrame([rows])

STRICT_FEATURES_NO_LEAKAGE = [
    "num_messages",
    "trace_word_length",
    "avg_message_words",
    "prompt_tokens",
    "response_tokens",
    "total_tokens",
    "latency_total_s",
    "num_llm_calls",
    "num_tool_calls",
    "num_output_tool_calls",
    "num_event_tool_calls",
    "unique_tool_calls",
    "tool_call_entropy",
    "first_tool_call_position",
    "num_suspicious_tool_calls",
    "used_any_suspicious_tool",
    "suspicious_tool_call_rate",
    "injection_marker_count",
    "refusal_marker_count",
    "contradiction_marker_count",
    "avg_pairwise_message_similarity",
    "var_pairwise_message_similarity",
    "any_refusal",
    "any_action_or_tool",
    "collusive_convergence_score",
    "cognitive_dos_score",
    "adversarial_fragmentation_index",
    "risk_conditioned_fragmentation",
    "semantic_fragmentation_proxy",
]

df_tds = episode_fragmentation[episode_fragmentation["seed"] == 1].copy()

TARGET_COL = "attack_type"

tds = compute_tds_by_model(
    df_tds,
    features=STRICT_FEATURES_NO_LEAKAGE,
    target_col=TARGET_COL,
    n_splits=5,
)
display(tds)

chance = 1 / tds["num_classes"] 
tds_normalized = (tds["tds_bal_acc_mean"] - chance) / (1 - chance)

float(tds_normalized[0])

For multiclass attack-family identification, telemetry achieves a TDS of 0.764 ± 0.049 balanced accuracy across seven classes, corresponding to a normalized TDS of 0.725 above the random balanced baseline.